# Plant-Wide Auto-Sequence Pipeline Trace

This notebook drives the **real** NexFeed plant-wide auto-sequence pipeline end-to-end and
prints a stage-by-stage trace.

It does **not** re-implement any pipeline logic in Python. Instead it calls a single
instrumented, **strictly read-only** server endpoint
(`POST /api/ai/auto-sequence/trace`) that runs the actual JavaScript pipeline modules
(the same code the browser runs) inside the dev server and returns a structured trace.

**9 stages (including Stage 5.5):**

1. Data Gathering
2. Combine + Line-Balance (pre-AI)
**5.5. Plant-Wide AI Load Rebalance** *(new — moves orders across lines to balance queue load before per-line AI runs)*
3. Per-line Pre-Sort
4. Prompt Construction
5. Azure OpenAI Call
6. Parse + Post-Process (AI rank vs final rank, divergence, faithfulness)
7. Metrics / Simulation
8. Result

> The endpoint is **read-only** — it only `SELECT`s from the database and runs pure
> compute. It never writes, updates, or deletes anything.

## 1. Configuration

In [11]:
# ── Run configuration ────────────────────────────────────────────────────────
import os

# Base URL of the running NexFeed dev server. In the Replit workspace the server
# listens on port 5000; you can also point at $REPLIT_DEV_DOMAIN.
BASE_URL = os.environ.get('NEXFEED_BASE_URL', 'http://127.0.0.1:5000')
TRACE_PATH = '/api/ai/auto-sequence/trace'
REQUEST_TIMEOUT = 180  # seconds — the Azure calls can take a while

# Plant-wide is the only supported scope for this trace (line/feedmill/upload
# auto-sequence are intentionally out of scope).
PLANT_SCOPE = 'plant'

# Data source: False = live tables (orders, next_10_days_*),
#              True  = isolated Fulfillment (Demo) dataset (demo_* tables).
USE_DEMO_DATA = False

# Show the verbatim system/user prompts and raw AI responses per line.
# Default ON; flip to False for a compact trace.
SHOW_RAW_PROMPTS = True

print('Target:      ', BASE_URL + TRACE_PATH)
print('Scope:       ', PLANT_SCOPE)
print('Data source: ', 'DEMO (demo_* tables)' if USE_DEMO_DATA else 'LIVE')
print('Raw prompts: ', 'shown' if SHOW_RAW_PROMPTS else 'hidden')

Target:       http://127.0.0.1:5000/api/ai/auto-sequence/trace
Scope:        plant
Data source:  LIVE
Raw prompts:  shown


## 2. Pretty-print helpers

In [12]:
import json, sys

# ── ANSI styling (degrades gracefully if the terminal has no color) ──────────
_C = {
    'reset': '\033[0m', 'bold': '\033[1m', 'dim': '\033[2m',
    'blue': '\033[94m', 'green': '\033[92m', 'yellow': '\033[93m',
    'red': '\033[91m', 'cyan': '\033[96m', 'magenta': '\033[95m',
}

def banner(text):
    line = '\u2500' * max(60, len(text) + 4)
    print(f"{_C['bold']}{_C['blue']}{line}{_C['reset']}")
    print(f"{_C['bold']}{_C['blue']}  {text}{_C['reset']}")
    print(f"{_C['bold']}{_C['blue']}{line}{_C['reset']}")

def info(text):
    print(f"{_C['cyan']}\u2139  {text}{_C['reset']}")

def ok(text):
    print(f"{_C['green']}\u2714  {text}{_C['reset']}")

def warn(text):
    print(f"{_C['yellow']}\u26a0  {text}{_C['reset']}")

def err(text):
    print(f"{_C['red']}\u2718  {text}{_C['reset']}")

def arrow(text):
    print(f"{_C['magenta']}   \u2192 {text}{_C['reset']}")

def show_json(obj, max_chars=2000, indent=2):
    """Collapsible-style JSON dump, truncated so big payloads don't flood the cell."""
    s = json.dumps(obj, indent=indent, default=str)
    if len(s) > max_chars:
        s = s[:max_chars] + f"\n... [truncated, {len(s) - max_chars} more chars]"
    for ln in s.splitlines():
        print(f"{_C['dim']}    {ln}{_C['reset']}")

def estimate_tokens(text):
    """Rough token estimate (~4 chars/token) matching the server-side heuristic."""
    if text is None:
        return 0
    return round(len(str(text)) / 4)

def fmt_ms(ms):
    if ms is None:
        return '   n/a'
    try:
        ms = float(ms)
    except (TypeError, ValueError):
        return '   n/a'
    if ms >= 1000:
        return f'{ms/1000:.2f}s'
    return f'{ms:.0f}ms'

ok('Helpers loaded.')

✔  Helpers loaded.


## 3. Call the trace endpoint

This runs the entire real pipeline once and returns the structured trace.

In [13]:
import json, time
import urllib.request
import urllib.error

def run_trace():
    url = BASE_URL + TRACE_PATH
    body = {'scope': PLANT_SCOPE, 'demo': bool(USE_DEMO_DATA)}
    data = json.dumps(body).encode('utf-8')
    req = urllib.request.Request(url, data=data, method='POST',
                                 headers={'Content-Type': 'application/json'})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=REQUEST_TIMEOUT) as resp:
            payload = json.loads(resp.read().decode('utf-8'))
    except urllib.error.HTTPError as e:
        body = e.read().decode('utf-8', 'replace')
        err(f'HTTP {e.code}: {body[:500]}')
        raise
    elapsed = time.time() - t0
    info(f'Round-trip: {elapsed:.1f}s')
    return payload

trace = run_trace()
if not trace.get('ok'):
    err('Trace failed: ' + str(trace.get('error')))
else:
    ok('Trace received.')
    meta = trace.get('meta', {})
    info(f"read-only: {trace.get('readOnly')}  |  orders: {meta.get('ordersTotal')}  |  lines: {len(meta.get('lines', []))}  |  totalMs: {meta.get('totalMs')}")

ℹ  Round-trip: 42.4s
✔  Trace received.
ℹ  read-only: True  |  orders: 56  |  lines: 7  |  totalMs: 41719


## 4. Stage-by-stage trace

Each stage prints its timing, token estimates (where relevant), and a collapsible JSON dump.
Stages are grouped by stage number, so Stage 6 shows both the raw parse and the
post-process artifacts (AI rank vs final rank, divergence, faithfulness).

In [14]:
from collections import OrderedDict

# Stage number -> (title, kind). kind drives how entries are rendered.
# Stage 5.5 uses the string key '5.5' because that is what the server emits
# (stageNo is a string for non-integer stages).
STAGE_META = OrderedDict([
    (1,     ('Stage 1 \u2014 Data Gathering',                     'global')),
    (2,     ('Stage 2 \u2014 Combine + Line-Balance (pre-AI)',     'global')),
    ('5.5', ('Stage 5.5 \u2014 Plant-Wide AI Load Rebalance',     'rebalance')),
    (3,     ('Stage 3 \u2014 Per-line Pre-Sort',                   'perline')),
    (4,     ('Stage 4 \u2014 Prompt Construction',                 'perline')),
    (5,     ('Stage 5 \u2014 Azure OpenAI Call',                   'perline')),
    (6,     ('Stage 6 \u2014 Parse + Post-Process',                'perline')),
    (7,     ('Stage 7 \u2014 Metrics / Simulation',                'perline')),
    (8,     ('Stage 8 \u2014 Result',                              'global')),
])

# Stage names whose JSON dumps contain the verbatim prompt / AI response.
RAW_PROMPT_STAGES = {'prompt_construction', 'azure_call'}

stages = trace.get('stages', [])
by_no = {}
for s in stages:
    by_no.setdefault(s.get('stageNo'), []).append(s)

def _entry_dump(entry):
    stage_name = entry.get('stage')
    data = entry.get('data', {})
    if stage_name in RAW_PROMPT_STAGES and not SHOW_RAW_PROMPTS:
        # Strip the big verbatim fields when raw prompts are hidden.
        slim = {k: v for k, v in (data or {}).items()
                if k not in ('systemPrompt', 'userPrompt', 'content')}
        slim['(raw hidden)'] = 'set SHOW_RAW_PROMPTS=True to view'
        show_json(slim, max_chars=1200)
    else:
        show_json(data, max_chars=1600)

def render_global_stage(entries):
    for e in entries:
        arrow(f"elapsed: {fmt_ms(e.get('elapsedMs'))}")
        _entry_dump(e)

def render_per_line_stage(entries):
    total = sum(float(e.get('elapsedMs') or 0) for e in entries)
    tok = sum(int(e.get('tokenEstimate') or 0) for e in entries)
    arrow(f"{len(entries)} entries | total {fmt_ms(total)}" + (f" | ~{tok} tokens" if tok else ''))
    for e in entries:
        head = f"{_C['bold']}{e.get('line','?')}{_C['reset']}"
        bits = [e.get('stage','?'), fmt_ms(e.get('elapsedMs'))]
        if e.get('tokenEstimate'):
            bits.append(f"~{e.get('tokenEstimate')} tok")
        if e.get('attempt') is not None:
            bits.append(f"attempt {e.get('attempt')}")
        print(f"  {head}  {_C['dim']}({', '.join(str(b) for b in bits)}){_C['reset']}")
        _entry_dump(e)

def render_rebalance_stage(entries):
    """Stage 5.5: Plant-Wide AI Load Rebalance renderer."""
    if not entries:
        warn('no Stage 5.5 entry found in trace')
        return
    e    = entries[0]
    data = e.get('data', {}) or {}
    elapsed  = fmt_ms(e.get('elapsedMs'))
    tok_est  = data.get('promptTokenEst', 0)

    if data.get('skipped'):
        warn(f"SKIPPED \u2014 {data.get('skipReason', 'unknown reason')}  (elapsed: {elapsed})")
        info('Stage 5.5 is skipped when skipAI=true is passed to the trace endpoint.')
        return

    if data.get('error'):
        err(f"ERROR \u2014 {data.get('error')}  (elapsed: {elapsed})")
        return

    divs = data.get('diversions', [])
    n    = data.get('diversionsApplied', len(divs))
    arrow(f"elapsed: {elapsed}  |  prompt tokens (est): ~{tok_est}  |  diversions applied: {n}")

    if not divs:
        ok('No diversions suggested \u2014 AI determined load is already balanced.')
        return

    # \u2500\u2500 Diversion table \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    W_IDX  = 3
    W_NAME = 50
    W_FROM = 9
    W_TO   = 9

    def _trunc(s, w):
        s = str(s)
        return (s[:w-1] + '\u2026') if len(s) > w else s

    sep = '  '.join(['-' * W_IDX, '-' * W_NAME, '-' * W_FROM, '-' * W_TO, '-' * 32])
    hdr = '  '.join([
        '#'.ljust(W_IDX), 'Order Name'.ljust(W_NAME),
        'From'.ljust(W_FROM), 'To'.ljust(W_TO), 'AI Reason',
    ])
    print(f"\n  {_C['bold']}{hdr}{_C['reset']}")
    print(f"  {sep}")
    for i, d in enumerate(divs, 1):
        row = '  '.join([
            str(i).ljust(W_IDX),
            _trunc(d.get('orderName', '?'), W_NAME).ljust(W_NAME),
            _trunc(d.get('fromLine',  '?'), W_FROM).ljust(W_FROM),
            _trunc(d.get('toLine',    '?'), W_TO  ).ljust(W_TO),
            str(d.get('reason', '?')),
        ])
        print(f"  {row}")
    print(f"  {sep}\n")

    # \u2500\u2500 Per-line order delta \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    from_counts = {}
    to_counts   = {}
    for d in divs:
        fl = d.get('fromLine', '?')
        tl = d.get('toLine',   '?')
        from_counts[fl] = from_counts.get(fl, 0) + 1
        to_counts[tl]   = to_counts.get(tl,   0) + 1
    all_lines = sorted(set(list(from_counts) + list(to_counts)))
    info('Per-line order delta from rebalancing:')
    for ln in all_lines:
        out = from_counts.get(ln, 0)
        inn = to_counts.get(ln, 0)
        net = inn - out
        sign = '+' if net > 0 else ''
        marker = ok if net > 0 else warn
        marker(f"{ln}:  \u2212{out} out  +{inn} in  \u2192  net {sign}{net} orders")

for no, (title, kind) in STAGE_META.items():
    entries = by_no.get(no, [])
    banner(title)
    if kind == 'rebalance':
        render_rebalance_stage(entries)
    elif not entries:
        warn('no entries for this stage')
    elif kind == 'global':
        render_global_stage(entries)
    else:
        render_per_line_stage(entries)
    print()

────────────────────────────────────────────────────────────
  Stage 1 — Data Gathering
────────────────────────────────────────────────────────────
   → elapsed: 323ms
    {
      "scope": "live",
      "ordersTable": "orders",
      "ordersTotal": 56,
      "kbSession": "kb_1779765959814",
      "kbRecords": 353,
      "n10dSession": "n10d-1780879740950",
      "n10dRecords": 21,
      "pmxSplitRules": 6,
      "inferredTargets": 21,
      "changeoverRulesSource": "dashboard_default",
      "changeoverRules": [
        {
          "id": "diameter_change",
          "title": "Change Pellet Diameter (4mm \u2194 3mm)",
          "reason": "Change Die",
          "icon": "\ud83d\udd35",
          "type": "diameter",
          "values": {
            "fm1": 1.5,
            "fm2": 1,
            "fm3": 1
          }
        },
        {
          "id": "yellow_brown",
          "title": "Change Color: Yellow \u2194 Brown",
          "reason": "Cleaning",
          "icon": "\ud83d\udfe1",


## 4b. Stage 5.5 — Rebalance Validation

This cell gives you three things to confirm Stage 5.5 really worked:

1. **Status** — was the stage reached, skipped, or did it error?
2. **Diversion table** — every order the AI moved, with the exact From/To lines and the AI’s one-sentence reason.
3. **Placement check** — verifies each diverted order *actually lands* on its target line in Stage 8’s final result, and is *absent* from the source line. This is the definitive proof the rebalance was applied.

In [ ]:
# ── Pull Stage 5.5 data ───────────────────────────────────────────────────────
rb_entries = by_no.get('5.5', [])
rb_data    = (rb_entries[0].get('data', {}) or {}) if rb_entries else {}

banner('Stage 5.5 — Plant-Wide AI Load Rebalance: Validation')

# ── 1. Status ────────────────────────────────────────────────────────────────
if not rb_entries:
    err('Stage 5.5 entry missing from trace — server did not emit it.')
    raise SystemExit

elapsed  = fmt_ms(rb_entries[0].get('elapsedMs'))
tok_est  = rb_data.get('promptTokenEst', 0)

if rb_data.get('skipped'):
    warn(f"Stage 5.5 SKIPPED — {rb_data.get('skipReason', '?')}")
    info('Re-run the trace without skipAI=true to exercise Stage 5.5.')
    raise SystemExit

if rb_data.get('error'):
    err(f"Stage 5.5 ERROR — {rb_data.get('error')}  (elapsed: {elapsed})")
    raise SystemExit

divs = rb_data.get('diversions', [])
n    = rb_data.get('diversionsApplied', len(divs))
ok(f'Stage 5.5 ran successfully  |  elapsed: {elapsed}  |  prompt tokens (est): ~{tok_est}')

# ── 2. Diversion table ───────────────────────────────────────────────────────
print()
if not divs:
    ok('AI returned 0 diversions — load already balanced across lines.')
else:
    info(f'{n} diversion(s) applied:')
    print()
    W_IDX  = 3
    W_NAME = 52
    W_FROM = 9
    W_TO   = 9
    W_RSN  = 60

    def _t(s, w):
        s = str(s)
        return (s[:w-1] + '…') if len(s) > w else s.ljust(w)

    sep = '+-' + '-+-'.join(['-'*W_IDX, '-'*W_NAME, '-'*W_FROM, '-'*W_TO, '-'*W_RSN]) + '-+'
    hdr = '| ' + ' | '.join([
        '#'.ljust(W_IDX), 'Order Name'.ljust(W_NAME),
        'From'.ljust(W_FROM), 'To'.ljust(W_TO),
        'AI Reason'.ljust(W_RSN)
    ]) + ' |'
    print(f"  {_C['bold']}{sep}{_C['reset']}")
    print(f"  {_C['bold']}{hdr}{_C['reset']}")
    print(f"  {_C['bold']}{sep}{_C['reset']}")
    for i, d in enumerate(divs, 1):
        row = '| ' + ' | '.join([
            _t(i,                        W_IDX),
            _t(d.get('orderName', '?'),  W_NAME),
            _t(d.get('fromLine',  '?'),  W_FROM),
            _t(d.get('toLine',    '?'),  W_TO),
            _t(d.get('reason',    '?'),  W_RSN),
        ]) + ' |'
        print(f'  {row}')
    print(f'  {sep}')
    print()

# ── 3. Placement check against Stage 8 result ───────────────────────────────
# Stage 8 (Result) holds the final per-line order lists after ALL stages.
# We verify: each diverted order appears on toLine and NOT on fromLine.
result_data  = (by_no.get(8, [{}])[0].get('data', {}) or {})
final_lines  = result_data.get('lines', {})   # { 'Line N': { ... } }

# Build orderId -> line mapping from Stage 8 if detail is available,
# else fall back to checking Stage 2 combined + Stage 3 pre-sort entries.
# Stage 8 'lines' contains 'ai1Orders' counts, not full order lists.
# So we use the Stage 8 per-line sequences embedded in the trace stages.
# The per-line result entries have stage='result' with data.sequence[]
placed_on = {}  # orderId -> line as recorded in final stage entries
for s in stages:
    if s.get('stage') == 'result':
        ln  = s.get('line')
        seq = (s.get('data') or {}).get('sequence', [])
        for o in seq:
            oid = str(o.get('orderId') or o.get('id', ''))
            if oid:
                placed_on[oid] = ln

print()
if not divs:
    info('No diversions to check.')
else:
    info('Placement check — confirming each diverted order reached its target line in the final result:')
    print()
    all_passed = True
    for d in divs:
        oid  = str(d.get('orderId', ''))
        name = d.get('orderName', oid)
        frm  = d.get('fromLine', '?')
        to   = d.get('toLine',   '?')
        actual = placed_on.get(oid)
        if actual is None:
            warn(f'  Order "{name}" ({oid}) — not found in any per-line result entry')
            info('  (Stage 8 may not include full order lists in this trace version — see JSON dump above)')
        elif actual == to:
            ok(f'  PASS  "{name}"  →  landed on {actual}  (expected {to})')
        else:
            err(f'  FAIL  "{name}"  →  found on {actual}  but expected {to}  (from {frm})')
            all_passed = False
    print()
    if placed_on:
        if all_passed:
            ok(f'All {n} diverted order(s) confirmed on their target lines. Stage 5.5 applied correctly.')
        else:
            err('One or more diverted orders did not reach their target line — check the trace above.')

## 5. Per-line result summary

Final recommended strategy + option order counts for each production line.

In [15]:
result_entries = by_no.get(8, [{}])
result_stage = result_entries[0] if result_entries else {}
lines = (result_stage.get('data', {}) or {}).get('lines', {})
banner('Per-line recommended strategies')
for ln, summ in lines.items():
    rec = summ.get('recommended')
    ok(f"{ln}: recommended={rec} | options={summ.get('optionCount')} "
       f"| ai1={summ.get('ai1Orders')} ai2={summ.get('ai2Orders')} rule={summ.get('ruleBasedOrders')}")

────────────────────────────────────────────────────────────
  Per-line recommended strategies
────────────────────────────────────────────────────────────
✔  Line 1: recommended=ai_option_1 | options=2 | ai1=8 ai2=0 rule=8
✔  Line 2: recommended=ai_option_1 | options=2 | ai1=6 ai2=0 rule=6
✔  Line 3: recommended=ai_option_1 | options=2 | ai1=10 ai2=0 rule=10
✔  Line 4: recommended=ai_option_1 | options=2 | ai1=7 ai2=0 rule=7
✔  Line 5: recommended=ai_option_1 | options=2 | ai1=4 ai2=0 rule=4
✔  Line 6: recommended=ai_option_2 | options=3 | ai1=6 ai2=6 rule=6
✔  Line 7: recommended=ai_option_1 | options=3 | ai1=4 ai2=4 rule=4


## 6. Token & timing roll-up

In [16]:
prompt_entries = [s for s in stages if s.get('stage') == 'prompt_construction']
azure_entries  = [s for s in stages if s.get('stage') == 'azure_call']

prompt_tokens = sum(int(e.get('tokenEstimate') or 0) for e in prompt_entries)
resp_tokens   = sum(int(e.get('tokenEstimate') or 0) for e in azure_entries)
azure_ms      = sum(float(e.get('elapsedMs') or 0) for e in azure_entries)

banner('Roll-up')
info(f'Prompt tokens (est):   {prompt_tokens}')
info(f'Response tokens (est): {resp_tokens}')
info(f'Total tokens (est):    {prompt_tokens + resp_tokens}')
info(f'Azure wall-time:       {fmt_ms(azure_ms)}')
info(f'End-to-end:            {fmt_ms(trace.get("meta", {}).get("totalMs"))}')

────────────────────────────────────────────────────────────
  Roll-up
────────────────────────────────────────────────────────────
ℹ  Prompt tokens (est):   22476
ℹ  Response tokens (est): 7372
ℹ  Total tokens (est):    29848
ℹ  Azure wall-time:       75.38s
ℹ  End-to-end:            41.72s


## 7. Read-only confirmation

The endpoint reports `readOnly: true` and performs only `SELECT` queries plus pure
compute against the real pipeline modules. Running this notebook never mutates
any orders, schedules, or master data.

In [17]:
assert trace.get('readOnly') is True, 'endpoint did not report read-only!'
ok('Confirmed: trace ran read-only \u2014 no data was modified.')

✔  Confirmed: trace ran read-only — no data was modified.
